In [1]:
import json
import random
import re
import time
from datetime import date, datetime, timezone
from enum import Enum
from pathlib import Path
from typing import Any

import pandas as pd
from pydantic import BaseModel, Field
from pydantic_ai import Agent

from renewables_permitting.utils import (
    normalize_text,
    save_parquet,
    validate_required_columns,
)

BASE_URL = "https://www.boe.es/datosabiertos/api/boe/sumario"

# PROJECT_ROOT = Path(__file__).resolve().parents[2]  # fuera del notebook
PROJECT_ROOT = Path.cwd().parent  # dentro del notebook

DATA_DIR = PROJECT_ROOT / "data"

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

# BRONZE
BOE_DOCS_XML_DIR = BRONZE_DIR / "boe_docs_xml"


# SILVER
BOE_CANDIDATES_PATH = SILVER_DIR / "boe_candidates" / "boe_candidates_normalized.parquet"

BOE_CANDIDATES_DOCS_TEXT_PATH = SILVER_DIR / "boe_candidates_docs_text" / "boe_candidates_docs_text.parquet"

DIM_MUNICIPALITIES_PATH = SILVER_DIR / "dimensions" / "dim_municipalities.parquet"

SILVER_BOE_AI_DIR = SILVER_DIR / "boe_ai"

BOE_AI_EXTRACTIONS_PATH = SILVER_BOE_AI_DIR / "boe_ai_extractions.parquet"
LIFECYCLE_EVENTS_PATH = SILVER_BOE_AI_DIR / "lifecycle_events.parquet"
ADMINISTRATIVE_ACTIONS_PATH = SILVER_BOE_AI_DIR / "administrative_actions.parquet"
ASSET_MENTIONS_PATH = SILVER_BOE_AI_DIR / "asset_mentions.parquet"
ASSET_TECHNOLOGIES_PATH = SILVER_BOE_AI_DIR / "asset_technologies.parquet"
ASSET_PARTICIPANTS_PATH = SILVER_BOE_AI_DIR / "asset_participants.parquet"
ASSET_LOCATIONS_PATH = SILVER_BOE_AI_DIR / "asset_locations.parquet"
ASSET_ALIASES_PATH = SILVER_BOE_AI_DIR / "asset_aliases.parquet"
ASSET_RELATION_MENTIONS_PATH = SILVER_BOE_AI_DIR / "asset_relation_mentions.parquet"


# GOLD
PROJECT_GROUPS_PATH = GOLD_DIR / "project_groups.parquet"
PROJECT_ASSETS_PATH = GOLD_DIR / "project_assets.parquet"
PROJECT_TIMELINE_PATH = GOLD_DIR / "project_timeline.parquet"
PROJECT_STATUS_PATH = GOLD_DIR / "project_status.parquet"

In [2]:
asset_locations = pd.read_parquet(ASSET_LOCATIONS_PATH)

In [3]:
asset_locations.loc[
    asset_locations["municipality_raw_norm"].str.contains("pontes", na=False)
]

,asset_location_id,asset_mention_id,event_id,identificador_boe,municipality_raw,municipality_raw_norm,province_hint_raw,province_hint_raw_norm,autonomous_community_hint_raw,autonomous_community_hint_raw_norm,location_evidence
35,BOE-B-2021-32560_event_1_asset_1_location_1,BOE-B-2021-32560_event_1_asset_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,As Pontes,as pontes,A Coruña,a coruna,None,None,"Municipios afectados: As Pontes, As Somozas, C..."
51,BOE-A-2023-2598_event_1_asset_1_location_6,BOE-A-2023-2598_event_1_asset_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,As Pontes de García Rodríguez,as pontes de garcia rodriguez,A Coruña,a coruna,None,None,As Pontes de García Rodríguez
57,BOE-A-2023-10306_event_1_asset_1_location_6,BOE-A-2023-10306_event_1_asset_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,As Pontés,as pontes,A Coruña,a coruna,Galicia,galicia,y As Pontés (A Coruña)
63,BOE-B-2023-19082_event_1_asset_1_location_6,BOE-B-2023-19082_event_1_asset_1,BOE-B-2023-19082_event_1,BOE-B-2023-19082,As Pontes de García Rodríguez,as pontes de garcia rodriguez,A Coruña,a coruna,Galicia,galicia,"términos municipales de Valdoviño, Cedeira, Ce..."
69,BOE-A-2024-16664_event_1_asset_1_location_6,BOE-A-2024-16664_event_1_asset_1,BOE-A-2024-16664_event_1,BOE-A-2024-16664,As Pontes de García Rodríguez,as pontes de garcia rodriguez,A Coruña,a coruna,None,None,ubicados en los términos municipales de Valdov...


Resolución geográfica robusta.

El resultado esperado es que todas esas filas acaben canonizadas como:

- municipality = Pontes de García Rodríguez, As
- municipality_norm = pontes de garcia rodriguez as
- ine_municipality_code = 15070
- province = Coruña, A
- ine_province_code = 15
- autonomous_community = Galicia
- ine_autonomous_community_code = 12

> TODO: El siguiente paso es crear asset_locations_resolved = resolve_asset_locations(asset_locations)

## 4. Herramientas del agente

### Localización

In [ ]:
municipios_ine_df = pd.read_parquet(DIM_MUNICIPALITIES_PATH)

In [ ]:
municipios_ine_df.loc[
    municipios_ine_df["municipio_norm"].str.contains("pontes", na=False),
    [
        "cauto",
        "cpro",
        "cmun",
        "comunidad_autonoma",
        "provincia",
        "municipio",
        "municipio_norm",
    ],
]

,cauto,cpro,cmun,comunidad_autonoma,provincia,municipio,municipio_norm
6965,12,15,070,Galicia,"Coruña, A","Pontes de García Rodríguez, As",pontes de garcia rodriguez as


In [ ]:
# Palabras con bajo valor discriminante para la resolución de municipios.
# Incluye preposiciones, contracciones, conjunciones y términos
# administrativos frecuentes en las distintas lenguas oficiales de España.

STOP_TOKENS = {
    # Preposiciones y contracciones
    "de", "del", "d", "da", "das", "do", "dos",
    "dels", "deth", "dera", "des", "en",

    # Conjunciones
    "y", "e", "i", "eta",

    # Términos administrativos
    "ayuntamiento", "municipio", "municipal",
    "termino", "término",
    "concello", "termo",
    "ajuntament", "municipi", "terme",
    "udal", "udala", "udalerri", "udalerria",
}

In [ ]:
def text_tokens(text: str | None) -> set[str]:
    """
    Convierte un texto en un conjunto de tokens normalizados,
    eliminando palabras poco informativas.
    """
    return {
        token
        for token in normalize_text(text).split()
        if token not in STOP_TOKENS
    }


def token_overlap_score(query: str | None, candidate: str | None) -> float:
    """
    Calcula la proporción de tokens de la consulta presentes
    en el candidato.

    Valor entre 0 y 1.
    """
    query_tokens = text_tokens(query)
    candidate_tokens = text_tokens(candidate)

    if not query_tokens or not candidate_tokens:
        return 0.0

    return len(query_tokens & candidate_tokens) / len(query_tokens)


def municipality_token_matches(
    municipality_name: str,
    candidate_municipality: str,
    *,
    allow_single_token: bool,
) -> bool:
    """
    Determina si un municipio candidato es compatible con la consulta.

    Reglas:
    - Coincidencia total de tokens -> match.
    - Coincidencia >= 80 % para consultas con varios tokens -> match.
    - Consultas de un solo token solo se aceptan si existen hints
      adicionales (provincia o comunidad autónoma).
    """
    query_tokens = text_tokens(municipality_name)
    candidate_tokens = text_tokens(candidate_municipality)

    if not query_tokens or not candidate_tokens:
        return False

    if query_tokens.issubset(candidate_tokens):
        return True

    if len(query_tokens) >= 2:
        return token_overlap_score(municipality_name, candidate_municipality) >= 0.8

    return allow_single_token and bool(query_tokens & candidate_tokens)


def hint_token_matches(
    hint: str | None,
    candidate: str | None,
) -> bool:
    """
    Comprueba si un hint administrativo (provincia o comunidad autónoma)
    comparte al menos un token relevante con el candidato.
    """
    if hint is None:
        return True

    hint_tokens = text_tokens(hint)
    candidate_tokens = text_tokens(candidate)

    if not hint_tokens or not candidate_tokens:
        return False

    return bool(hint_tokens & candidate_tokens)


def _row_to_location(row: pd.Series) -> MunicipalityLocation:
    """
    Convierte una fila de la dimensión INE en un objeto tipado.
    """
    return MunicipalityLocation(
        municipality=row["municipio"],
        province=row["provincia"],
        autonomous_community=row["comunidad_autonoma"],
        ine_municipality_code=row["cpro"] + row["cmun"],
        ine_province_code=row["cpro"],
        ine_autonomous_community_code=row["cauto"],
    )


def _build_lookup_result(
    municipality_name: str,
    province_hint: str | None,
    autonomous_community_hint: str | None,
    matches: pd.DataFrame,
    matched_by: str,
    reason: str,
) -> MunicipalityLookupResult:
    """
    Construye la respuesta final a partir de las coincidencias obtenidas.

    - 1 coincidencia  -> RESOLVED
    - >1 coincidencia -> AMBIGUOUS
    - 0 coincidencias -> NOT_FOUND
    """
    matches = matches.drop_duplicates(
        subset=["cauto", "cpro", "cmun"]
    )

    if len(matches) == 1:
        return MunicipalityLookupResult(
            query=municipality_name,
            province_hint=province_hint,
            autonomous_community_hint=autonomous_community_hint,
            resolution_status=MunicipalityResolutionStatus.RESOLVED,
            resolved=_row_to_location(matches.iloc[0]),
            matched_by=matched_by,
            reason=reason,
        )

    if len(matches) > 1:
        return MunicipalityLookupResult(
            query=municipality_name,
            province_hint=province_hint,
            autonomous_community_hint=autonomous_community_hint,
            resolution_status=MunicipalityResolutionStatus.AMBIGUOUS,
            candidates=[_row_to_location(row) for _, row in matches.iterrows()],
            matched_by=matched_by,
            reason=(
                "Existen varias coincidencias compatibles con los criterios "
                "proporcionados."
            ),
        )

    return MunicipalityLookupResult(
        query=municipality_name,
        province_hint=province_hint,
        autonomous_community_hint=autonomous_community_hint,
        resolution_status=MunicipalityResolutionStatus.NOT_FOUND,
        reason="No existe coincidencia en el catálogo INE.",
    )


def resolve_municipality_impl(
    municipality_name: str,
    province_hint: str | None = None,
    autonomous_community_hint: str | None = None,
) -> MunicipalityLookupResult:
    """
    Resuelve un municipio español contra la dimensión INE.

    Orden de resolución:
    1. Coincidencia por nombres normalizados de búsqueda.
    2. Desambiguación por provincia.
    3. Desambiguación por comunidad autónoma.
    4. Coincidencia flexible por tokens como fallback.
    """

    municipality_name_norm = normalize_text(municipality_name)
    province_hint_norm = normalize_text(province_hint) if province_hint else None
    autonomous_community_hint_norm = (
        normalize_text(autonomous_community_hint)
        if autonomous_community_hint
        else None
    )

    if "municipio_lookup_names_norm" not in municipios_ine_df.columns:
        raise ValueError(
            "La dimensión de municipios no contiene la columna "
            "'municipio_lookup_names_norm'. Reejecuta 06_tablas_referencia_ine.ipynb."
        )

    # Fase 1: coincidencia por nombre oficial normalizado o variante válida.
    matches = municipios_ine_df.loc[
        municipios_ine_df["municipio_lookup_names_norm"].map(
            lambda names: municipality_name_norm in names
        )
    ]

    # Fase 2: desambiguar mediante provincia.
    if not matches.empty and province_hint_norm:
        province_matches = matches.loc[
            matches["provincia"].map(
                lambda value: hint_token_matches(province_hint, value)
            )
        ]

        if not province_matches.empty:
            return _build_lookup_result(
                municipality_name=municipality_name,
                province_hint=province_hint,
                autonomous_community_hint=autonomous_community_hint,
                matches=province_matches,
                matched_by="municipality_lookup_name_and_province_hint",
                reason=(
                    "Municipio resuelto por nombre normalizado de búsqueda "
                    "y provincia compatible por tokens."
                ),
            )

    # Fase 3: desambiguar mediante comunidad autónoma.
    if not matches.empty and autonomous_community_hint_norm:
        ac_matches = matches.loc[
            matches["comunidad_autonoma"].map(
                lambda value: hint_token_matches(
                    autonomous_community_hint,
                    value,
                )
            )
        ]

        if not ac_matches.empty:
            return _build_lookup_result(
                municipality_name=municipality_name,
                province_hint=province_hint,
                autonomous_community_hint=autonomous_community_hint,
                matches=ac_matches,
                matched_by="municipality_lookup_name_and_autonomous_community_hint",
                reason=(
                    "Municipio resuelto por nombre normalizado de búsqueda "
                    "y comunidad autónoma compatible por tokens."
                ),
            )

    # Fase 4: resolver si la coincidencia por nombres de búsqueda es única;
    # si hay varias, devolver AMBIGUOUS.
    if not matches.empty:
        return _build_lookup_result(
            municipality_name=municipality_name,
            province_hint=province_hint,
            autonomous_community_hint=autonomous_community_hint,
            matches=matches,
            matched_by="municipality_lookup_name",
            reason=(
                "Municipio resuelto por nombre oficial normalizado o variante "
                "normalizada de búsqueda."
            ),
        )

    # Fase 5: búsqueda flexible por tokens como fallback.
    allow_single_token = (
        province_hint is not None
        or autonomous_community_hint is not None
    )

    partial_matches = municipios_ine_df.loc[
        municipios_ine_df["municipio"].map(
            lambda value: municipality_token_matches(
                municipality_name,
                value,
                allow_single_token=allow_single_token,
            )
        )
    ]

    # Fase 6: filtrar coincidencias parciales mediante provincia.
    if not partial_matches.empty and province_hint is not None:
        province_partial_matches = partial_matches.loc[
            partial_matches["provincia"].map(
                lambda value: hint_token_matches(province_hint, value)
            )
        ]

        if not province_partial_matches.empty:
            return _build_lookup_result(
                municipality_name=municipality_name,
                province_hint=province_hint,
                autonomous_community_hint=autonomous_community_hint,
                matches=province_partial_matches,
                matched_by="municipality_token_and_province_hint",
                reason=(
                    "Municipio resuelto por coincidencia de tokens del municipio "
                    "y provincia compatible por tokens."
                ),
            )

    # Fase 7: filtrar coincidencias parciales mediante comunidad autónoma.
    if not partial_matches.empty and autonomous_community_hint is not None:
        ac_partial_matches = partial_matches.loc[
            partial_matches["comunidad_autonoma"].map(
                lambda value: hint_token_matches(
                    autonomous_community_hint,
                    value,
                )
            )
        ]

        if not ac_partial_matches.empty:
            return _build_lookup_result(
                municipality_name=municipality_name,
                province_hint=province_hint,
                autonomous_community_hint=autonomous_community_hint,
                matches=ac_partial_matches,
                matched_by="municipality_token_and_autonomous_community_hint",
                reason=(
                    "Municipio resuelto por coincidencia de tokens del municipio "
                    "y comunidad autónoma compatible por tokens."
                ),
            )

    # Fase 8: devolver coincidencias parciales restantes.
    if not partial_matches.empty:
        return _build_lookup_result(
            municipality_name=municipality_name,
            province_hint=province_hint,
            autonomous_community_hint=autonomous_community_hint,
            matches=partial_matches,
            matched_by="municipality_token",
            reason=(
                "Existen coincidencias por tokens del municipio, pero no hay "
                "hints suficientes para garantizar una resolución única."
            ),
        )

    # Fase 9: sin coincidencias.
    return MunicipalityLookupResult(
        query=municipality_name,
        province_hint=province_hint,
        autonomous_community_hint=autonomous_community_hint,
        resolution_status=MunicipalityResolutionStatus.NOT_FOUND,
        reason=(
            "No existe coincidencia por nombre normalizado de búsqueda "
            "ni por tokens en el catálogo INE."
        ),
    )

resolve_municipality() es una herramienta disponible para el modelo, pero no una garantía determinista. El modelo puede:

1. llamar a la herramienta y copiar bien la respuesta;
2. llamar a la herramienta y copiarla mal;
3.no llamar a la herramienta;
4. completar campos administrativos por su cuenta;
5. mezclar una variante textual del municipio con códigos inventados o arrastrados.

In [ ]:
@agent.tool_plain
def resolve_municipality(
    municipality_name: str,
    province_hint: str | None = None,
    autonomous_community_hint: str | None = None,
) -> MunicipalityLookupResult:
    return resolve_municipality_impl(
        municipality_name=municipality_name,
        province_hint=province_hint,
        autonomous_community_hint=autonomous_community_hint,
    )

# resolve_municipality("Amurrio")
# resolve_municipality("Agurain/Salvatierra")
# resolve_municipality("Palmas")
# resolve_municipality("Palmas", province_hint="Las Palmas")
# resolve_municipality("Gran Canaria", province_hint="Las Palmas")
# resolve_municipality("Palmas", autonomous_community_hint="Canarias")

## Flatten

In [ ]:
def flatten_asset_locations(ai_extractions: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in ai_extractions.iterrows():
        extraction = BOEProjectExtraction.model_validate_json(row["extraction_json"])

        for event_idx, event in enumerate(extraction.lifecycle_events, start=1):
            event_id = f"{extraction.identificador_boe}_event_{event_idx}"

            for asset in event.assets:
                asset_mention_id = f"{event_id}_{asset.local_asset_id}"

                for location_idx, loc in enumerate(asset.locations, start=1):
                    records.append(
                        {
                            "asset_location_id": f"{asset_mention_id}_location_{location_idx}",
                            "asset_mention_id": asset_mention_id,
                            "event_id": event_id,
                            "identificador_boe": extraction.identificador_boe,
                            "municipality": loc.municipality,
                            "municipality_norm": normalize_text(loc.municipality) if loc.municipality else None,
                            "province": loc.province,
                            "province_norm": normalize_text(loc.province) if loc.province else None,
                            "autonomous_community": loc.autonomous_community,
                        }
                    )

    return pd.DataFrame(records)